# Amun — Breath Signal Exploration

The original *Invisible-Driver* fit an **unsupervised clustering model** over EEG
features (reporting a silhouette score). Amun does the honest equivalent for
**breath loudness**: we cluster recorded frames into *silence / soft / hard* and
report the **measured** silhouette score.

Everything here runs on the **bundled, synthetic** sample dataset
(`data/breath_labelled.csv`) so it works with **no microphone**. Numbers are
computed live — nothing is hard-coded.

In [ ]:
import sys, csv
from pathlib import Path
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

rows = list(csv.DictReader(open(ROOT / 'data' / 'breath_labelled.csv')))
loud = np.array([float(r['loudness']) for r in rows])
labels = [r['label'] for r in rows]
print(f'{len(rows)} frames:', {l: labels.count(l) for l in sorted(set(labels))})

In [ ]:
# Per-state loudness statistics (the anchors calibration learns)
for state in ['silence', 'soft', 'hard']:
    x = loud[[l == state for l in labels]]
    print(f'{state:8s}  mean={x.mean():.4f}  std={x.std():.4f}  '
          f'range=[{x.min():.4f}, {x.max():.4f}]')

In [ ]:
# Histogram of the three breath states
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
for state, color in [('silence', '#37e0d8'), ('soft', '#f5c451'), ('hard', '#ff6b9d')]:
    x = loud[[l == state for l in labels]]
    ax.hist(x, bins=30, alpha=0.7, label=state, color=color)
ax.set_xlabel('loudness (RMS)'); ax.set_ylabel('frames'); ax.legend()
ax.set_title('Breath loudness distribution by state'); plt.tight_layout(); plt.show()

In [ ]:
# Fit Amun's actual calibration model and report the MEASURED silhouette
from amun.classify import fit_profile
profile = fit_profile(loud.tolist())
print('learned anchors:')
print(f'  noise_floor = {profile.noise_floor}')
print(f'  soft        = {profile.soft}')
print(f'  hard        = {profile.hard}')
print(f'\nmeasured silhouette score = {profile.silhouette}  (on bundled sample data)')

The three breath states separate cleanly, which is why a simple k-means(3) gives
a strong silhouette. In the live game the same three anchors normalise your
breath into a continuous lift force — see `src/amun/classify.py` and
`src/amun/preprocessing.py`.

> **Honesty note:** this score is measured on synthetic sample data designed to
> imitate real breath statistics. Your own calibration (run `amun calibrate` or
> the in-browser step) will produce a score for *your* microphone and breathing.